# TicketIQ Exploratory Data Analysis (EDA)

This notebook performs preliminary analysis on the processed training set ():
1. **Class Balance**: Operational Category & Urgency distributions
2. **Text Length Distribution**: Character lengths and word counts
3. **Top TF-IDF Terms per Category**: Most informative unigrams and bigrams per class


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("../../data/processed/train.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")


Dataset Shape: 16622 rows, 13 columns
Columns: ['ticket_id', 'text', 'subject', 'body', 'answer', 'type', 'queue', 'category', 'priority', 'urgency', 'created_at', 'char_length', 'word_count']


## 1. Class Balance

In [2]:
print("=== CATEGORY DISTRIBUTION (4-CLASS) ===")
cat_counts = df["category"].value_counts()
cat_pcts = (cat_counts / len(df) * 100).round(2)
print(pd.DataFrame({"Count": cat_counts, "Percent (%)": cat_pcts}))
print("")
print("=== URGENCY DISTRIBUTION (3-CLASS) ===")
urg_counts = df["urgency"].value_counts()
urg_pcts = (urg_counts / len(df) * 100).round(2)
print(pd.DataFrame({"Count": urg_counts, "Percent (%)": urg_pcts}))


=== CATEGORY DISTRIBUTION (4-CLASS) ===
                       Count  Percent (%)
category                                 
Technical Support       5456        32.82
Customer Service        3246        19.53
Product Support         3100        18.65
IT Support              2305        13.87
Billing and Payments    1694        10.19
Returns and Exchanges    821         4.94

=== URGENCY DISTRIBUTION (3-CLASS) ===
         Count  Percent (%)
urgency                    
Medium    6828        41.08
High      6403        38.52
Low       3391        20.40


## 2. Text Length Distribution

In [3]:
df["char_length"] = df["text"].fillna("").str.len()
df["word_count"] = df["text"].fillna("").apply(lambda x: len(x.split()))
print("=== CHARACTER LENGTH STATS ===")
print(df["char_length"].describe().round(2))
print("")
print("=== WORD COUNT STATS ===")
print(df["word_count"].describe().round(2))


=== CHARACTER LENGTH STATS ===
count    16622.00
mean       415.21
std        207.25
min         18.00
25%        246.00
50%        410.00
75%        573.00
max       1825.00

=== WORD COUNT STATS ===
count    16622.00
mean        61.18
std         31.97
min          3.00
25%         35.00
50%         60.00
75%         84.00
max        276.00


## 3. Top TF-IDF Terms Per Category

In [4]:
vec = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X = vec.fit_transform(df["text"].fillna(""))
feature_names = np.array(vec.get_feature_names_out())
print("=== TOP TF-IDF TERMS PER CATEGORY ===")
for cat in sorted(df["category"].unique()):
    mask = (df["category"] == cat).values
    cat_mean = np.asarray(X[mask].mean(axis=0)).flatten()
    top_indices = np.argsort(cat_mean)[::-1][:10]
    terms = [f"{feature_names[i]} ({cat_mean[i]:.4f})" for i in top_indices]
    print(f"[{cat}]")
    print("  " + ", ".join(terms))
    print("")


=== TOP TF-IDF TERMS PER CATEGORY ===
[Billing and Payments]
  billing (0.1070), payment (0.0672), information (0.0327), data (0.0308), services (0.0288), details (0.0282), subscription (0.0280), support (0.0275), provide (0.0259), options (0.0250)

[Customer Service]
  data (0.0497), investment (0.0307), analytics (0.0278), digital (0.0270), strategies (0.0269), integration (0.0266), medical (0.0265), information (0.0257), security (0.0256), support (0.0254)

[IT Support]
  data (0.0420), software (0.0293), issue (0.0292), problem (0.0248), security (0.0243), support (0.0241), analytics (0.0236), tools (0.0231), medical (0.0224), digital (0.0216)

[Product Support]
  data (0.0478), issue (0.0287), security (0.0280), integration (0.0252), analytics (0.0249), medical (0.0244), investment (0.0238), support (0.0234), problem (0.0232), digital (0.0230)

[Returns and Exchanges]
  data (0.0498), investment (0.0355), security (0.0324), software (0.0248), digital (0.0241), marketing (0.0230), 